# Détection de routes — GPSO

Pipeline : **WMS IGN → FLAIR-INC → masque routes → raster commune → publication Portal**

| Étape | Module |
|-------|--------|
| Téléchargement ortho-photos | `ortho_ign` |
| Segmentation sémantique | `flair_inference` |
| Logique métier routes | ce notebook |
| Publication Portal | `arcgis` |

Communes disponibles : `BOULOGNE-BILLANCOURT`, `CHAVILLE`, `ISSY-LES-MOULINEAUX`,
`MARNES-LA-COQUETTE`, `MEUDON`, `SEVRES`, `VANVES`, `VILLE-D'AVRAY`

## 1 · Installation & modèle

In [ ]:
import subprocess, sys, zipfile, urllib.request, pathlib

subprocess.run(
    ["conda", "install", "-c", "conda-forge", "-y", "--quiet",
     "rasterio", "geopandas", "shapely", "fiona", "onnxruntime", "huggingface_hub"],
    check=True,
)

LIB_DIR = pathlib.Path("/arcgis/home/landcover-lib")
if not LIB_DIR.exists():
    zip_path = pathlib.Path("/arcgis/home/landcover.zip")
    urllib.request.urlretrieve(
        "https://github.com/DIGIT-Seine-Ouest/cv-routes-bitumees/archive/refs/heads/main.zip",
        zip_path,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(LIB_DIR)

sys.path.insert(0, str(next(LIB_DIR.iterdir())))

## 2 · Configuration

In [ ]:
from flair_inference import download_model, FlairModel

# --- À adapter ---
COMMUNE    = "MEUDON"
TILES_DIR  = f"/arcgis/home/tiles/{COMMUNE}"
EXPORT_DIR = f"/arcgis/home/exports/{COMMUNE}"

# Classes FLAIR qui définissent une "route"
ROAD_CLASSES = {1, 2}        # pervious_surface + impervious_surface
ROAD_COLOR   = (255, 140, 0)

GPSO_GEOJSON_URL = (
    "https://data.seineouest.fr/api/explore/v2.1/catalog/datasets/plu-de-gpso"
    "/exports/geojson?lang=fr&timezone=Europe%2FBerlin&epsg=2154"
)

model = FlairModel(download_model(dest=f"/arcgis/home/flair_12cl_resnet34_unet.onnx"))
print(f"Modèle prêt — {model.NUM_CLASSES} classes FLAIR")

## 3 · Téléchargement des tuiles WMS

In [ ]:
from ortho_ign import fetch_city_tiles

def progress(r, m):
    print(f"\r  {'█' * int(r * 30) + '░' * (30 - int(r * 30))}  {m}", end="", flush=True)

tiles = fetch_city_tiles(
    commune_name=COMMUNE,
    tiles_dir=TILES_DIR,
    geojson_url=GPSO_GEOJSON_URL,
    on_progress=progress,
)
print(f"\n{len(tiles)} tuile(s) prête(s)")

## 4 · Inférence FLAIR

In [ ]:
import numpy as np
from flair_inference import mask_from_classes, colorize, apply_overlay, class_stats
from ortho_ign import load_tile_as_rgb
from PIL import Image

results = []

for i, tile_path in enumerate(tiles):
    progress(i / len(tiles), f"Tuile {i + 1}/{len(tiles)}")
    img    = load_tile_as_rgb(tile_path)
    arr255 = np.transpose(np.array(img, dtype=np.float32), (2, 0, 1))

    class_map = model.predict(arr255)
    road_mask = mask_from_classes(class_map, ROAD_CLASSES)
    stats     = class_stats(class_map, target_ids=ROAD_CLASSES)

    results.append({
        "tile_path":    tile_path,
        "ortho":        img,
        "flair_map":    Image.fromarray(colorize(class_map)),
        "road_overlay": apply_overlay(img, road_mask, ROAD_COLOR),
        "road_mask":    road_mask,
        "road_pct":     stats["target_pct"],
    })

avg_road = sum(r["road_pct"] for r in results) / len(results)
print(f"\n{len(results)} tuiles — surface route moyenne : {avg_road:.1f}%")

## 5 · Visualisation rapide

In [ ]:
import matplotlib.pyplot as plt

n = min(3, len(results))
fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
if n == 1: axes = [axes]

for i, r in enumerate(results[:n]):
    axes[i][0].imshow(r["ortho"])
    axes[i][1].imshow(r["flair_map"])
    axes[i][2].imshow(r["road_overlay"])
    axes[i][2].set_title(f"{r['road_pct']:.1f}%", fontsize=9)
    for ax in axes[i]: ax.axis("off")

axes[0][0].set_title("Ortho IGN")
axes[0][1].set_title("FLAIR — 12 classes")
axes[0][2].set_title("Routes détectées")
plt.suptitle(COMMUNE, fontsize=13)
plt.tight_layout()
plt.show()

## 6 · Création du raster commune

On fusionne les masques de toutes les tuiles en **un seul GeoTIFF** couvrant la commune entière (EPSG:2154).
C'est ce fichier qui sera publié sur le Portal.

In [ ]:
import os
import rasterio
from rasterio.merge import merge as rio_merge
from ortho_ign import mask_to_geotiff

os.makedirs(EXPORT_DIR, exist_ok=True)
tif_dir = os.path.join(EXPORT_DIR, "tuiles")
os.makedirs(tif_dir, exist_ok=True)

# 1. Masque par tuile
tif_paths = []
for r in results:
    tile_name = os.path.splitext(os.path.basename(r["tile_path"]))[0]
    tif_paths.append(mask_to_geotiff(
        r["road_mask"], r["tile_path"],
        os.path.join(tif_dir, f"{tile_name}_routes.tif"),
    ))

# 2. Fusion → raster commune entière (un seul GeoTIFF, pas de gaps)
MERGED_TIF = os.path.join(EXPORT_DIR, f"{COMMUNE.lower()}_routes.tif")
sources = [rasterio.open(p) for p in tif_paths]
mosaic, transform = rio_merge(sources)
meta = sources[0].meta.copy()
meta.update(count=1, width=mosaic.shape[2], height=mosaic.shape[1],
            transform=transform, compress="lzw", nodata=255)
for src in sources:
    src.close()
with rasterio.open(MERGED_TIF, "w", **meta) as dst:
    dst.write(mosaic[0:1])

print(f"Raster : {MERGED_TIF}  ({os.path.getsize(MERGED_TIF)/1e6:.1f} MB)")
print(f"CRS    : {meta['crs']}  |  {meta['width']}×{meta['height']} px")

## 7 · Publication sur ArcGIS Portal

Upload du raster fusionné et publication comme **Hosted Imagery Layer**.

In [ ]:
from arcgis.gis import GIS

gis = GIS("home")
print(f"Connecté : {gis.properties.portalHostname}  ({gis.users.me.username})")

In [ ]:
ITEM_TITLE = f"Routes FLAIR-INC — {COMMUNE}"
ITEM_TAGS  = "FLAIR, routes, GPSO, raster, détection, IGN"
ITEM_DESC  = (
    "Raster binaire de détection de routes produit par FLAIR-INC ResNet34-UNet (IGN France). "
    "Valeurs : 1 = route (imperméable + perméable), 0 = non-route. "
    "Résolution source : 0.20 m/px (BD ORTHO® IGN). CRS : EPSG:2154."
)

folder = gis.content.folders.get()

# Upload du GeoTIFF
print("Upload du raster…")
tif_item = folder.add(
    item_properties={
        "title":       ITEM_TITLE,
        "type":        "GeoTIFF",
        "tags":        ITEM_TAGS,
        "snippet":     f"Masque routes FLAIR-INC — {COMMUNE} (GPSO) — EPSG:2154",
        "description": ITEM_DESC,
    },
    file=MERGED_TIF,
).result()
print(f"  item id : {tif_item.id}")

# Publication comme Hosted Imagery Layer
print("Publication comme couche raster…")
imagery_item = tif_item.publish(
    publish_parameters={"name": f"routes_flair_{COMMUNE.lower()}"}
)

print(f"\n✓ Couche publiée : {imagery_item.title}")
print(f"  URL            : {imagery_item.homepage}")
imagery_item